# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Dataset metadata as an object
md = dataset.metadata
print(md.name)
print(md.description)

# Optionally, print more metadata fields
print("\nPublished:", getattr(md, 'datePublished', 'N/A'))
print("Keywords:", getattr(md, 'keywords', 'N/A'))


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their '@id' and name
print('Available record sets:')
for rs in md.recordSet:
    print(f"  @id={rs['@id']}, name={rs.get('name', '')}")

# Choose the primary record set for analysis
# There may be just one main record set, so we will select the first one
if len(md.recordSet) > 0:
    main_rs = md.recordSet[0]
    print(f"\nFields and columns in the record set '@id': {main_rs['@id']}")

    # List all fields
    for field in main_rs['field']:
        print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}, column: {field.get('column', {}).get('@id') if isinstance(field.get('column'), dict) else field.get('column')}")
else:
    print('No record sets found in the dataset metadata!')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define record set(s) to extract using @id
record_sets = [rs['@id'] for rs in md.recordSet]
dataframes = {}

for record_set_id in record_sets:
    # Use mlcroissant to extract records
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns and first rows for the first record set
primary_rs_id = record_sets[0] if record_sets else None
if primary_rs_id is not None:
    print('RecordSet @id:', primary_rs_id)
    print('Columns:', dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())
else:
    print('No data extracted.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis by '@id' or display all numeric columns
import numpy as np
df = dataframes[primary_rs_id]
print('Numeric columns:')
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_columns)

# For demonstration, select the first available numeric field
if numeric_columns:
    numeric_field = numeric_columns[0]
    threshold = df[numeric_field].quantile(0.75)  # e.g., upper quartile

    # Filter rows where the numeric field is greater than the threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold}:")
    display(filtered_df.head())

    # Normalize this numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optionally, group by a categorical field (if any)
    cat_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_columns:
        group_field = cat_columns[0]
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean of '{numeric_field}' grouped by '{group_field}':")
        display(grouped.head())
else:
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot distribution of the chosen numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of "{numeric_field}"')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Optionally, boxplot by group_field
    if cat_columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'Boxplot of {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using `mlcroissant`. We loaded the metadata, reviewed available record sets and fields (referenced by their `@id`), extracted the main data, performed exploratory analysis including filtering and normalization on numeric fields, and visualized key data distributions. For deeper research questions, you can adapt the above workflow by referencing other record sets, fields, and analyzing relevant columns using their `@id` for traceability and reproducibility.